### Agent Evaluation

For RAG, we used the A->Q->A' setup:

- A = original answer in the FAQ
- Q = generated question from this answer
- A' = answer produced by our RAG system

For agents, we use the same setup. A' comes from an agent instead of a
fixed RAG pipeline.

We also save the trajectory. Here, the trajectory means only the tool
calls the agent made before producing the final answer.

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
import sys
sys.path.append("..")

In [3]:
from src.ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

Create a lookup table:

In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

### Running the agent

In [5]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

Define the search tool:

In [6]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

Create the runner:

In [7]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [8]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

In [10]:
print(result)

LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None), EasyInputMessage(content='I just found this course — is it still okay to join now, or am I too late?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"late to join course still okay to join now enrollment after start FAQ"}', call_id='call_fh44fI4ytML4fuf5Om3NLU9K', name='search', type='function_call', id='fc_02fb119b940f0a97006a9d0b8f4b1087d2a6a6f74bfead3c21', caller=None, namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_fh44fI4ytML4fuf5Om3NLU9K', 'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a cer

In [11]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='I just found this course — is it still okay to join now, or am I too late?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"late to join course still okay to join now enrollment after start FAQ"}', call_id='call_fh44fI4ytML4fuf5Om3NLU9K', name='search', type='function_call', id='fc_02fb119b940f0a97006a9d0b8f4b1087d2a6a6f74bfead3c21', caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_fh44fI4ytML4fuf5Om3NLU9K',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you nee

For this lesson, the trajectory is only the tool calls. We don't need to send the full message history to the judge.

Extract the function name and arguments:

In [12]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [13]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"late to join course still okay to join now enrollment after start FAQ"}'}]

Get the original answer:

In [14]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

Save the A->Q->A' record and the trajectory:

In [16]:
import json

agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": json.dumps(tool_calls),
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'I just found this course — is it still okay to join now, or am I too late?',
 'answer_agent': 'Yes — you can still join.\n\nAccording to the course FAQ, if you just discovered the course, that’s fine. You can start learning whenever you want, and you can submit homework as long as the submission form is still open. The main caveat is that if you want a certificate, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': '[{"name": "search", "arguments": "{\\"query\\":\\"late to join course still okay to join now enrollment after start FAQ\\"}"}]',
 'cost': Decimal('0.00120375'),
 'document': '74eb249bbf'}

### Processing multiple questions

Create a function that processes one ground truth record:

In [18]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": json.dumps(tool_calls),
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

Run it for a small sample in parallel:

In [19]:
from concurrent.futures import ThreadPoolExecutor
from src.evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=5) as pool:
    agent_answers = map_progress(pool, ground_truth[:15], generate_agent_answer)

  0%|          | 0/15 [00:00<?, ?it/s]

In [23]:
df_agent = pd.DataFrame(agent_answers)
print(df_agent.shape)
df_agent.head()

(15, 6)


,question,answer_agent,answer_orig,tool_calls,cost,document
0,I just found this course — is it still okay to...,Yes — you can still join now. According to the...,"Yes, but if you want to receive a certificate,...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.0011955,74eb249bbf
1,Can I still enroll if I discovered the course ...,Yes — you can still join even if you discovere...,"Yes, but if you want to receive a certificate,...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.0013515,74eb249bbf
2,"If I join the course late, can I still get a c...","Yes — you can still join late, but to get a ce...","Yes, but if you want to receive a certificate,...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.0011475,74eb249bbf
3,What do I need to do to be eligible for the ce...,"If you start now, you can still work through t...","Yes, but if you want to receive a certificate,...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.0015645,74eb249bbf
4,Is it fine to take the course after it has beg...,Yes — you can start the course whenever you wa...,"Yes, but if you want to receive a certificate,...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.0020625,74eb249bbf


In [21]:
df_agent["cost"].sum()

Decimal('0.01981725')

In [22]:
df_agent.to_csv("agent-answers.csv", index=False)

### Judging answers and trajectories

A good trajectory is not just "many tool calls". A good trajectory uses
the available tools in a way that helps answer the question.

For our search agent, a good trajectory has these properties:

- The search query is relevant to the user question
- The query includes the important keywords from the question
- The agent avoids duplicate searches with the same arguments
- If it searches more than once, the next query is a useful refinement
- It usually uses 1 search call
- 2-3 calls can be okay for harder questions
- More than 3 search calls needs a clear reason
- The tool calls support the final answer
- The agent does not stop too early or keep searching without a reason

Now define a judge output type with two scores:

In [24]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

The judge instructions:

In [25]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

Define the judge function:

In [27]:
from src.evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [28]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning='The agent’s answer matches the ground truth. It correctly says the learner can still join now, and it includes the important certificate condition: the project must be submitted while submissions are still being accepted. It also mentions homework submission timing, which is extra but not contradictory to the original answer.', answer_score='good', trajectory_reasoning='The search query was relevant to the user’s question about whether it is too late to join a course. It included key concepts like joining late/too late/still okay to join now. Only one search was used, which is reasonable for a straightforward FAQ-style question, and the tool call supported the final answer.', trajectory_score='good')

When the answer is bad, the trajectory score tells us whether the problem started with tool use. If the answer is bad but the trajectory is good, the model may have used the retrieved context poorly. If both are bad, the agent likely searched for the wrong thing. It may also have stopped too early.

### Running the agent judge

In [29]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [30]:
with ThreadPoolExecutor(max_workers=5) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/15 [00:00<?, ?it/s]

In [31]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [33]:
df_agent_eval = pd.DataFrame(agent_evaluations)
print(df_agent_eval.shape)
df_agent_eval.head()

(15, 6)


,question,document,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
0,I just found this course — is it still okay to...,74eb249bbf,good,The agent’s answer matches the ground truth: i...,good,The tool call was reasonable and relevant. The...
1,Can I still enroll if I discovered the course ...,74eb249bbf,good,The agent's answer matches the ground truth. I...,good,The search query was relevant to the question ...
2,"If I join the course late, can I still get a c...",74eb249bbf,good,The agent answer matches the ground truth: it ...,good,The search query was relevant to the question ...
3,What do I need to do to be eligible for the ce...,74eb249bbf,bad,The agent’s answer does not match the ground t...,good,The search query was somewhat relevant because...
4,Is it fine to take the course after it has beg...,74eb249bbf,bad,The agent’s answer is only partially aligned w...,good,The search queries were relevant to the genera...


In [34]:
calc_total_price(usages)

0.01663725

In [35]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    13
bad      2
Name: count, dtype: int64

In [36]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    15
Name: count, dtype: int64

In [37]:
df_agent_eval.to_csv("agent-evaluations.csv", index=False)